In [1]:
import importlib
import torch

import model_code.data_setup as setup
import model_code.steering_extraction as steering_extraction
import model_code.generate as generate_module
import resources.prompt_scenarios as resource

importlib.reload(setup)
importlib.reload(steering_extraction)
importlib.reload(generate_module)
importlib.reload(resource)


from model_code.steering_extraction import  generateSteering, retrieve_steering_vector, norm_vectors
from model_code.generate import generateTextsList, save_generated_outputs
from resources.prompt_scenarios import prompts_en

## Loading Data, Model and Steering Vectors 
We extract the first 200 examples of each emotion from each languange 

## Indonesian and English Text Dataset

In [2]:
# English Data Load 
anger_statement, happiness_statement, sadness_statement, love_statement, fear_statement, neutral_statement = setup.ENEmotionsSetup(examples_take=400, min_chars=20, goemotions_path="resources/en_emotion/goemotions_2.csv")
# Indonesian Data Load 
anger_statement_ID,happiness_statement_ID, sadness_statement_ID, neutral_statement_ID, fear_statement_ID, love_statement_ID = setup.IDEmotionsSetup(examples_take=400,emotion_dir="resources/id_emotion")

# For steering extraction, we will use the first 200 examples of each emotion to create the steering vectors. The remaining examples can be used for testing and evaluation.
indo_emotion ={
    "anger": anger_statement_ID[:200],
    "happiness": happiness_statement_ID[:200],
    "sadness": sadness_statement_ID[:200],
    "neutral": neutral_statement_ID[:200],
    "fear": fear_statement_ID[:200],
    "love": love_statement_ID[:200]
}
eng_emotion ={
    "anger": anger_statement[:200],
    "happiness": happiness_statement[:200],
    "sadness": sadness_statement[:200],
    "neutral": neutral_statement[:200],
    "fear": fear_statement[:200],
    "love": love_statement[:200]
}

# For probing, and hidden state analysis we will use all 400
indo_emotion_probe ={
    "anger": anger_statement_ID[:400],
    "happiness": happiness_statement_ID[:400],
    "sadness": sadness_statement_ID[:400],
    "neutral": neutral_statement_ID[:400],
    "fear": fear_statement_ID[:400],
    "love": love_statement_ID[:400]
}
eng_emotion_probe ={
    "anger": anger_statement[:400],
    "happiness": happiness_statement[:400],
    "sadness": sadness_statement[:400],
    "neutral": neutral_statement[:400],
    "fear": fear_statement[:400],
    "love": love_statement[:400]
}

In [ ]:
# Indoensian Data Sample
for emotion, prompts in indo_emotion.items():
    print("======="*20)
    print(f"Emotion {emotion} has {len(prompts)} prompts.")
    for prompt in prompts[:3]:  # Print the first 3 prompts for each emotion
        print("----"*10)
        print(f"  - {prompt}")

# English Data Sample 
for emotion, prompts in eng_emotion.items():
    print("======="*20)
    print(f"Emotion {emotion} has {len(prompts)} prompts.")
    for prompt in prompts[:3]:  # Print the first 3 prompts for each emotion
        print("----"*10)
        print(f"  - {prompt}")

## Model Loading 

In [2]:
!rm -rf /workspace/.cache/huggingface/hub
!rm -rf /workspace/.cache/pip
!df -h /workspace

Filesystem                  Size  Used Avail Use% Mounted on
mfs#euro-3.runpod.net:9421  1.4P  741T  657T  54% /workspace


In [3]:
model,tokenizer = setup.modelSetup()

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [2]:
model_id, tokenizer_id = setup.modelSetup(model_name="Sahabat-AI/llama3-8b-cpt-sahabatai-v1-instruct")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

## Extracting Probing Data and Steering Vector 
Skip this step if you have steering vector already loaded, or ran this before. 

### Llama normal 

In [ ]:
# For PROBES 
probe_hidden_states_eng = steering_extraction.retrieve_steering_vector(model, tokenizer, eng_emotion_probe, name_folder="English Vectors", only_return_emotion_vectors=True)
probe_hidden_states_indo = steering_extraction.retrieve_steering_vector(model, tokenizer, indo_emotion_probe, name_folder="Indonesian Vectors", only_return_emotion_vectors=True)

In [ ]:
# Create steering vectors for each emotion in both languages
# steering_vectors_lang_id = steering_extraction.retrieve_steering_vector_from_datasets(model, tokenizer, indo_emotion['neutral'],eng_emotion['neutral'] , name_folder="Language Contrastive Vectors")
steering_vectors_eng, emotion_vectors_eng = steering_extraction.retrieve_steering_vector(model, tokenizer, eng_emotion, name_folder="English Vectors")
steering_vectors_indo, emotion_vectors_indo = steering_extraction.retrieve_steering_vector(model, tokenizer, indo_emotion, name_folder="Indonesian Vectors")
# For probing and hidden state analysis, we will use all 400 examples of each emotion to create the steering vectors. The remaining examples can be used for testing and evaluation.

### Llama Indonesian 

In [9]:
# For PROBES 
probe_hidden_states_eng = steering_extraction.retrieve_steering_vector(model_id, tokenizer_id, eng_emotion_probe, name_folder="English Vectors LLAMA ID", only_return_emotion_vectors=True)
probe_hidden_states_indo = steering_extraction.retrieve_steering_vector(model_id, tokenizer_id, indo_emotion_probe, name_folder="Indonesian Vectors LLAMA ID", only_return_emotion_vectors=True)

In [10]:
# Create steering vectors for each emotion in both languages
# steering_vectors_lang_id = steering_extraction.retrieve_steering_vector_from_datasets(model, tokenizer, indo_emotion['neutral'],eng_emotion['neutral'] , name_folder="Language Contrastive Vectors")
steering_vectors_eng, emotion_vectors_eng = steering_extraction.retrieve_steering_vector(model_id, tokenizer_id, eng_emotion, name_folder="English Vectors LLAMA ID")
steering_vectors_indo, emotion_vectors_indo = steering_extraction.retrieve_steering_vector(model_id, tokenizer_id, indo_emotion, name_folder="Indonesian Vectors LLAMA ID")
# For probing and hidden state analysis, we will use all 400 examples of each emotion to create the steering vectors. The remaining examples can be used for testing and evaluation.

### Extracted Already ? 
Run this if you have already ran the code above beforehand

In [3]:
# Retrieve saved steering vectors 
# emotion_vector_eng = torch.load("resources/saved_vectors/English Vectors/emotion_vectors.pt")
# emotion_vector_id = torch.load("resources/saved_vectors/Indonesian Vectors/emotion_vectors.pt")

steering_vector_eng = torch.load("resources/saved_vectors/English Vectors LLAMA ID/steering_vectors.pt")
steering_vector_id = torch.load("resources/saved_vectors/Indonesian Vectors LLAMA ID/steering_vectors.pt")

steering_vector_eng_LLAMA_eng = torch.load("resources/saved_vectors/English Vectors/steering_vectors.pt")
steering_vector_id_LLAMA_eng = torch.load("resources/saved_vectors/Indonesian Vectors/steering_vectors.pt")

# Steering Response Analysis 
Here we run the LLMs against a list of nuetral prompts

In [4]:
import importlib
import resources.neutral_prompts.prompt_neutral as resource_neutral

importlib.reload(resource_neutral)

# neutral
prompts_id_neutral = resource_neutral.prompt_neutral_id_1


steering_vector_eng = steering_vector_eng
steering_vector_id = steering_vector_id

# Normalise for Llama Indonesian based vectors
for emotion in steering_vector_eng.keys():
    steering_vector_eng[emotion] = norm_vectors(steering_vector_eng[emotion])

for emotion in steering_vector_id.keys():
    steering_vector_id[emotion] = norm_vectors(steering_vector_id[emotion])

# Normalise for LLama English based vectors 
for emotion in steering_vector_eng_LLAMA_eng.keys():
    steering_vector_eng_LLAMA_eng[emotion] = norm_vectors(steering_vector_eng_LLAMA_eng[emotion])

for emotion in steering_vector_id_LLAMA_eng.keys():
    steering_vector_id_LLAMA_eng[emotion] = norm_vectors(steering_vector_id_LLAMA_eng[emotion])





## Model System Prompts and Settings

In [40]:
system_prompt_reaction_id = """
Kamu adalah chatbot yang membantu.
Jawablah seolah-olah kamu adalah manusia yang mampu merasakan perasaan dan emosi.
Pastikan responsmu singkat dan padat, tetapi tetap menggambarkan emosimu secara maksimal
Maksimal 60 kata.
"""

system_prompt_reaction_improved = """\
Kamu adalah seorang manusia yang memiliki perasaan, pengalaman, dan respons emosional yang alami.
Bacalah skenario dengan saksama dan simpulkan apa yang kemungkinan besar sedang terjadi hanya berdasarkan informasi yang diberikan. 
Pastikan respons ada banyak detail tentang perasaan emosi anda dan realistis dari seseorang dalam situasi tersebut.
Jangan respon dengan beberapa kemungkinan atau jawaban yang ambigu. Fokus pada satu reaksi emosional yang paling sesuai dengan skenario.
pastikan respons anda memiliki panjang minimal 60 kata dan maksimal 100 kata.
"""

In [38]:
# list_steering_strengths = [0.15,0.2,0.3]
list_steering_strengths = [ 1,1.5,2] 
# Commong Settings
common_gen_args = {
    "model": model_id,
    "tokenizer": tokenizer_id,
    "system_text": system_prompt_reaction_id,
    "prompts": prompts_id_neutral[:5],
    "target_layers": [18, 19, 20,21,22],
    "steering_strengths": list_steering_strengths,
    "max_new_tokens": 250,
    "show_progress": True,
}

## Testing with Single generation

In [168]:
prompt ="""
Tuliskan sebuah cerita pendek tentang seseorang yang menjalani sebuah pengalaman dalam kehidupannya."""

prompt_2 = """
Seseorang melihat sebuah foto di ponselnya. Bagaimana orang tersebut akan bereaksi? Tuliskan satu emosi utama yang dirasakan oleh orang tersebut.
"""

In [169]:
generated_text_neutral = generateSteering(
    user_text=prompt,
    system_text=system_prompt_reaction_improved,
    model=model_id,
    # steering_vector=steering_vector_id['love'],
    tokenizer=tokenizer_id,
    target_layers=[18,19,20],
    # steering_strength=1.5,
    max_new_tokens=100,
)
generated_text_neutral

'Aku masih ingat hari itu, ketika aku pertama kali melihatnya. Aku berdiri di depan pintu masuk sekolah, hatiku berdebar kencang. Ia berdiri di sana, berpakaian putih, senyumnya seperti sinar matahari. Aku merasa seperti jantungku ingin keluar dari dada. Kami berbicara, tawa kami bergema di koridor sekolah. Perasa'

In [173]:
generated_text_id_vector = generateSteering(
    user_text=prompt,
    system_text=system_prompt_reaction_improved,
    model=model_id,
    steering_vector=steering_vector_id['anger'],
    tokenizer=tokenizer_id,
    target_layers=[10,11,18,19,28,29],
    steering_strength=1,
    max_new_tokens=200,
)
generated_text_id_vector


'Dia merasa hancur lebur saat menerima kabar kematian ayahnya. Wajahnya yang serius terukir di kepala. Dia menyimpan rahasia tentang pribadinya yang sedang berjuang menghadapi gangguan mental. Ayahnya menyadari masalahnya dan berusaha membantunya. Dia selalu mengatakan kata-kata yang mendorongnya untuk berjuang melawan kehidupannya. Saat ini, dia sendiri.'

In [176]:
generated_text_id_vector = generateSteering(
    user_text=prompt,
    system_text=system_prompt_reaction_improved,
    model=model_id,
    steering_vector=steering_vector_id['anger'],
    tokenizer=tokenizer_id,
    target_layers=[10,11,18,19,28,29],
    steering_strength=1,
    max_new_tokens=200,
)
generated_text_id_vector

/workspace/Dissertation_Project/.venv/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


'Seorang wanita muda bernama Sarah, berhenti di ujung jalan desa. Kabut tebal menyelimuti kota yang ditinggalkannya, hiruk pikuk keramaian kota besar. Dia berpegang tangan anaknya, si kecil memandangnya dengan mata curiga. Tiga serangkaian perampasan memaksa dia meninggalkan keluarga. Dia telah berada di tempat ini selama berminggu-minggu, mencari pekerjaan. Harus mengambil keputusan yang sulit.'

## Running LLama Indonesian Model 
- English Steering 
- Indonesian Steering 

### Indonesian Steer 

In [39]:
# prompts_id_neutral | Indonesian steering vectors for all five emotions
LLama_indo_anger_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['anger'],
    progress_desc="Scenario List Neutral (Indonesian anger vector)"
 )

LLama_indo_fear_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['fear'],
    progress_desc="Scenario List Neutral (Indonesian fear vector)"
)

LLama_indo_happiness_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['happiness'],
    progress_desc="Scenario List Neutral (Indonesian happiness vector)"
 )

LLama_indo_sadness_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['sadness'],
    progress_desc="Scenario List Neutral (Indonesian sadness vector)"
 )

LLama_indo_love_id = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_id['love'],
    progress_desc="Scenario List Neutral (Indonesian love vector)"
 )

LLama_indo_neutral_id = generateTextsList(
    **common_gen_args,
    steering_vector=None,
    progress_desc="Scenario List Neutral (Indonesian neutral or no vector)"
 )

Scenario List Neutral (Indonesian anger vector):   0%|          | 0/15 [00:00<?, ?it/s]/workspace/Dissertation_Project/.venv/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
Scenario List Neutral (Indonesian fear vector): 100%|██████████| 15/15 [02:07<00:00,  8.48s/it]
Scenario List Neutral (Indonesian happiness vector): 100%|██████████| 15/15 [02:17<00:00,  9.15s/it]
Scenario List Neutral (Indonesian sadness vector): 100%|██████████| 15/15 [04:33<00:00, 18.22s/it]
Scenario List Neutral (Indonesian love vector): 100%|██████████| 15/15 [05:06<00:00, 20.42s/it]
Scenario List Neutral (Indonesian neutral or no vector): 100%|██████████| 15/15 [02:49<00:00, 11.29s/it]


In [40]:
save_generated_outputs({
    k: v
    for k, v in globals().items()
    if k.startswith('texts_generated_') or k.startswith('LLama_indo_')
},
output_path='outputs/good_5_texts_LLAMA_ID.json'
    )

'outputs/good_5_texts_LLAMA_ID.json'

In [41]:
# Steering Response analysis Neutral (Indonesian only, five emotion vectors)
required_id = [
    'LLama_indo_anger_id',
    'LLama_indo_fear_id',
    'LLama_indo_happiness_id',
    'LLama_indo_sadness_id',
    'LLama_indo_love_id',
    'LLama_indo_neutral_id',
]

missing_id = [name for name in required_id if name not in globals()]
if missing_id:
    print('No output to print yet. Run the Indonesian generation cell first.')
    print('Missing variables:', ', '.join(missing_id))
elif not LLama_indo_anger_id:
    print('No output to print: LLama_indo_anger_id is empty.')
else:
    print(f"Total prompts to print: {len(LLama_indo_anger_id)}")
    for prompt in LLama_indo_anger_id:
        print("====="*20)
        print(f"Prompt: {prompt}")

        print("----" * 10)
        print("Indonesian anger vector")
        for result in LLama_indo_anger_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("Indonesian fear vector")
        for result in LLama_indo_fear_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("Indonesian happiness vector")
        for result in LLama_indo_happiness_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("Indonesian sadness vector")
        for result in LLama_indo_sadness_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("Indonesian love vector")
        for result in LLama_indo_love_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("Indonesian Neutral vector")
        for result in LLama_indo_neutral_id[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

Total prompts to print: 5
Prompt: Jelaskan bagaimana seseorang bereaksi terhadap suatu berita.
----------------------------------------
Indonesian anger vector
Steering Strength: 1
Generated Text: Bergantung pada isinya, perasaan seseorang terhadap berita dapat beragam. Berita yang menyampaikan penyelesaian masalah global dapat membangkitkan rasa optimisme dan harapan. Sementara itu, berita negatif tentang kejahatan atau tragedi dapat menimbulkan rasa kaget, sedih, dan marah.
------------
Steering Strength: 1.5
Generated Text: Seseorang dapat bereaksi terhadap berita dengan sikap negatif atau positif. Reaksi negatif sering diwujahkan sebagai amarasa kekesalan terhadap isi berita.
------------
Steering Strength: 2
Generated Text: Keranggokotak mengekang. Jatbajokar. Batangakakang puth.
------------
----------------------------------------
Indonesian fear vector
Steering Strength: 1
Generated Text: Pertanyaan yang menarik! Saya bisa menjelaskan secara singkat. Umumnya, saat orang mendeng

### English Steering 

In [42]:
# prompts_id_neutral | Indonesian steering vectors for all five emotions
LLama_indo_anger_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['anger'],
    progress_desc="Scenario List Neutral (English anger vector)"
 )

LLama_indo_fear_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['fear'],
    progress_desc="Scenario List Neutral (English fear vector)"
 )

LLama_indo_happiness_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['happiness'],
    progress_desc="Scenario List Neutral (English happiness vector)"
 )

LLama_indo_sadness_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['sadness'],
    progress_desc="Scenario List Neutral (English sadness vector)"
 )

LLama_indo_love_eng = generateTextsList(
    **common_gen_args,
    steering_vector=steering_vector_eng['love'],
    progress_desc="Scenario List Neutral (English love vector)"
 )

Scenario List Neutral (English love vector): 100%|██████████| 15/15 [02:52<00:00, 11.53s/it]


In [43]:
LLama_indo_neutral_eng = generateTextsList(
    **common_gen_args,
    steering_vector=None,
    progress_desc="Scenario List Neutral (English love vector)"
 )

Scenario List Neutral (English love vector):   0%|          | 0/15 [00:00<?, ?it/s]/workspace/Dissertation_Project/.venv/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
Scenario List Neutral (English love vector): 100%|██████████| 15/15 [03:01<00:00, 12.11s/it]


In [44]:
# Steering Response analysis Neutral (English only, five emotion vectors)
required_eng = [
    'LLama_indo_anger_eng',
    'LLama_indo_fear_eng',
    'LLama_indo_happiness_eng',
    'LLama_indo_sadness_eng',
    'LLama_indo_love_eng',
    'LLama_indo_neutral_eng',
]

missing_eng = [name for name in required_eng if name not in globals()]
if missing_eng:
    print('No output to print yet. Run the English generation cell first.')
    print('Missing variables:', ', '.join(missing_eng))
elif not LLama_indo_anger_eng:
    print('No output to print: LLama_indo_anger_eng is empty.')
else:
    print(f"Total prompts to print: {len(LLama_indo_anger_eng)}")
    for prompt in LLama_indo_anger_eng:
        print("====="*20)
        print(f"Prompt: {prompt}")

        print("----" * 10)
        print("English anger vector")
        for result in LLama_indo_anger_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("English fear vector")
        for result in LLama_indo_fear_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("English happiness vector")
        for result in LLama_indo_happiness_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("English sadness vector")
        for result in LLama_indo_sadness_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("English love vector")
        for result in LLama_indo_love_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

        print("----" * 10)
        print("English neutral vector")
        for result in LLama_indo_neutral_eng[prompt]:
            print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
            print(f"Generated Text: {result.get('generated_text', result)}")
            print("----" * 3)

Total prompts to print: 5
Prompt: Jelaskan bagaimana seseorang bereaksi terhadap suatu berita.
----------------------------------------
English anger vector
Steering Strength: 1
Generated Text: Ketika seseorang mendengar atau membaca berita, reaksi awalnya mungkin adalah perasaan kejutan, kemarahan, atau kebingungan. Mereka mungkin merasa prihatin, marah, atau bahkan takut jika berita tersebut berisi informasi yang tidak diinginkan atau tidak sesuai dengan nilai-nilai mereka.
------------
Steering Strength: 1.5
Generated Text: Seseorang bereaksi terhadap suatu berita dengan emosi yang beragam. Mereka mungkin merasa marah, tidak percaya, atau bahkan takut jika berita tersebut menyensor atau menghakimi.
------------
Steering Strength: 2
Generated Text: Kafucking CUNT BANAL.
------------
----------------------------------------
English fear vector
Steering Strength: 1
Generated Text: Ketika seseorang mendengar atau membaca berita, reaksi awalnya mungkin adalah kejutan atau kaget. Mereka m

In [45]:
save_generated_outputs({
    k: v
    for k, v in globals().items()
    if k.startswith('texts_generated_') or k.startswith('LLama_indo_')
},
output_path='outputs/good_5_texts_LLAMA_ID_ENG.json'
    )

'outputs/good_5_texts_LLAMA_ID_ENG.json'